# Segmental-duplication percentage across species

Computes the fraction of each genome covered by segmental duplications, from
BISER `segdup_output_<species>` files (in
`code/command-line-script/genome-annotation/biser/<species>/`). For each species it
counts the genome length, removes reciprocal duplicate links, merges overlapping
alignments, and reports `segdup_perc = seg_dup_length / genome_length * 100`.


In [ ]:
# Project root — edit for your environment.
PROJ_ROOT = "/tscc/projects/ps-renlab2/jhc103/degu-genome-assembly-proj"


In [ ]:
import pandas as pd
import numpy as np


In [ ]:
# ============================================================================
# Helper functions
# ============================================================================
def count_bases(fasta_file):
    """Total genome length (bp) from a FASTA file."""
    length = 0
    with open(fasta_file) as f:
        for line in f:
            if not line.startswith(">"):
                length += len(line.strip())
    print(f"Total genome length: {length:,} bp")
    return length

def remove_reciprocal_duplicates(df):
    """Remove reciprocal duplicate links, keeping one representation of each pair."""
    seen = set()
    keep = []
    for _, row in df.iterrows():
        norm = tuple(sorted([
            (row["chr1"], row["start1"], row["end1"]),
            (row["chr2"], row["start2"], row["end2"]),
        ]))
        if norm not in seen:
            seen.add(norm)
            keep.append(True)
        else:
            keep.append(False)
    cleaned = df[keep].reset_index(drop=True)
    print(f"Removed {len(df) - len(cleaned):,} reciprocal duplicates ({len(df):,} -> {len(cleaned):,} rows)")
    return cleaned

def merge_alignments(df, chrom_col="chr1", start_col="start1", end_col="end1"):
    """Merge overlapping/adjacent alignments per chromosome; return merged intervals with lengths."""
    merged_data = []
    for chrom, group in df.groupby(chrom_col):
        sorted_group = group.sort_values(start_col)
        current_start = sorted_group.iloc[0][start_col]
        current_end = sorted_group.iloc[0][end_col]
        for _, row in sorted_group.iloc[1:].iterrows():
            start, end = row[start_col], row[end_col]
            if start <= current_end:
                current_end = max(current_end, end)
            else:
                merged_data.append({"chrom": chrom, "start": current_start, "end": current_end,
                                    "length": current_end - current_start})
                current_start, current_end = start, end
        merged_data.append({"chrom": chrom, "start": current_start, "end": current_end,
                            "length": current_end - current_start})
    return pd.DataFrame(merged_data)


In [ ]:
# ============================================================================
# Inputs: BISER output dir + species -> genome-fasta mapping
# ============================================================================
input_dir = f"{PROJ_ROOT}/code/command-line-script/genome-annotation/biser"

species_dict = {
    "dog": "GCF_000002285.5_Dog10K_Boxer_Tasha_genomic.fna",
    "domesticated_guinea_pig": "GCF_034190915.1_mCavPor4.1_genomic.fna",
    "Egyptian_jerboa": "GCF_020740685.1_mJacJac1.mat.Y.cur_genomic.fna",
    "Egyptian_spiny_rat": "GCA_029890205.1_ASM2989020v1_genomic.fna",
    "groundhog": "GCF_021218885.2_Marmota_monax_Labrador192_F-V1.1_genomic.fna",
    "hifiasm-041425": "deNovo_hifiHiCMode_hifiData_aggressivePurge3_kmer21_041325.asm.hic.p_ctg.fa",
    "house_mouse": "GCF_000001635.27_GRCm39_genomic.fna",
    "human": "GCF_000001405.40_GRCh38.p14_genomic.fna",
    "Long-tailed_chinchilla": "GCF_000276665.1_ChiLan1.0_genomic.fna",
    "Naked_mole_rat": "GCA_944319725.1_Naked_mole-rat_paternal_genomic.fna",
    "Norway_rat": "GCF_036323735.1_GRCr8_genomic.fna",
    "octDeg1": "genome.fa",
    "pacific_pocket_mouse": "GCF_023159225.1_ASM2315922v1_genomic.fna",
    "Siberian_hamster": "GCA_030556225.1_Psun-UiT-1_genomic.fna",
}


In [ ]:
# ============================================================================
# Calculate segdup percentage for every species
# ============================================================================
BEDPE_COLS = ["chr1", "start1", "end1", "chr2", "start2", "end2", "reference", "score",
              "strand1", "strand2", "max_len", "aln_len", "cigar", "optional"]

rows = []
for key in species_dict:
    print(f"=== {key} ===")
    genome_len = count_bases(f"{input_dir}/{key}/{species_dict[key]}")
    dup = pd.read_csv(f"{input_dir}/{key}/segdup_output_{key}", sep="\t", header=None, names=BEDPE_COLS)
    dup = remove_reciprocal_duplicates(dup)
    merged_dup = merge_alignments(dup)
    seg_dup_len = merged_dup["length"].sum()
    rows.append({"species": key, "genome_length": genome_len, "seg_dup_length": seg_dup_len})

df = pd.DataFrame(rows)
df["segdup_perc"] = (df["seg_dup_length"] / df["genome_length"]) * 100
df
